In [20]:
# Import Libraries
from bs4 import BeautifulSoup
import requests

import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.geometry import Point

import os
from os import path
import json
import time
import random
from pathlib import Path
from dotenv import load_dotenv
from scipy.spatial import cKDTree


In [48]:
# Get file names 
hdb_points_file = "resale_2019_to_2025_cleaned.geojson"
output_file = "hdb_mall_distances.csv"

### 1. Function to get list of shopping malls in Singapore

In [50]:
url = "https://en.wikipedia.org/wiki/Category:Shopping_malls_in_Singapore"
response = requests.get(url, timeout = 5)
content = BeautifulSoup(response.content, "html.parser") # parse HTML content

In [51]:
def extract_mall_names(url):
    headers = {"User-Agent": "Mozilla/5.0"}

    response = requests.get(url, headers=headers, timeout=30)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    # Wikipedia category pages usually put the page list here
    pages_div = soup.find("div", id="mw-pages")
    if pages_div is None:
        raise ValueError("Could not find the category pages section.")

    malls = []

    # Get all links inside the category listing
    for a in pages_div.select("a"):
        name = a.get_text(strip=True)
        href = a.get("href", "")

        # Skip non-article items
        if not href.startswith("/wiki/"):
            continue
        if ":" in href.replace("/wiki/", "", 1):   
            continue
        if name == "List of shopping malls in Singapore":
            continue

        malls.append(name)

    return sorted(set(malls))

In [52]:
malls = extract_mall_names(url)
malls = [mall.removesuffix(", Singapore") for mall in malls]
print(f"Extracted {len(malls)} mall names")

Extracted 110 mall names


### 2. Get mall coordinates using OneMap API 

In [81]:
def get_coordinates_onemap(place_name):
    base_url = "https://www.onemap.gov.sg/api/common/elastic/search"

    params = {
        "searchVal": place_name,
        "returnGeom": "Y",
        "getAddrDetails": "Y",
        "pageNum": 1
    }

    resp = requests.get(base_url, params=params, timeout=30)
    time.sleep(0.5)
    resp.raise_for_status()

    data = resp.json()
    results = data.get("results", [])

    if not results:
        return None

    best = results[0]

    return {
        "name": place_name,
        "matched_name": best.get("BUILDING") or best.get("SEARCHVAL") or place_name,
        "address": best.get("ADDRESS"),
        "postal": best.get("POSTAL"),
        "latitude": float(best["LATITUDE"]),
        "longitude": float(best["LONGITUDE"])
    }


def malls_to_geodataframe(malls):
    """
    Convert a list of mall names into:
    - GeoDataFrame of matched malls
    - list of missing / unmatched malls
    """
    rows = []
    missing = []

    for mall in malls:
        mall = mall.title()
        try:
            result = get_coordinates_onemap(mall)
            if result is None:
                missing.append(mall.upper())
            else:
                rows.append(result)
        except Exception as e:
            missing.append(mall.upper())
            print(f"Failed for {mall}: {e}")

    if not rows:
        empty_gdf = gpd.GeoDataFrame(
            columns=["matched_name", "latitude", "longitude", "geometry"],
            geometry="geometry",
            crs="EPSG:4326"
        )
        return empty_gdf, missing

    df = pd.DataFrame(rows)

    gdf = gpd.GeoDataFrame(
        df,
        geometry=[Point(xy) for xy in zip(df["longitude"], df["latitude"])],
        crs="EPSG:4326"
    )

    return gdf, missing

In [82]:
mall_gdf, missing_malls = malls_to_geodataframe(malls)
print("\nMissing malls:", missing_malls)


Missing malls: ['CAPITOL CENTRE', 'CHANGE ALLEY', 'ENG WAH GLOBAL', 'I12 KATONG', 'INTERNATIONAL PLAZA (SINGAPORE)', 'SHAW HOUSE AND CENTRE', 'WHITE SANDS SHOPPING MALL']


### 3. Get walking distance between closest mall and HDB 

In [28]:
load_dotenv()

url = "https://www.onemap.gov.sg/api/auth/post/getToken"

payload = {
    "email": os.getenv("ONEMAP_EMAIL"),
    "password": os.getenv("ONEMAP_EMAIL_PASSWORD")
}

response = requests.post(url, json=payload)

if response.status_code == 200:
    data = response.json()
    onemap_api_key = data.get("access_token")  
else:
    onemap_api_key = None
    print("Failed to get token")

3.1 Run a sample of postal codes to ensure code works, check the result output to see if there are any nulls 

In [29]:
# --- 1. load and prep data ---
raw_hdb_gdf = gpd.read_file(hdb_points_file).to_crs(epsg=4326)
# take only the first 5 unique postals for this sample
postal_gdf_sample = raw_hdb_gdf.drop_duplicates(subset=['postal_code']).head(5).copy()

mall_gdf = mall_gdf.to_crs(epsg=4326)

mall_coords = list(zip(mall_gdf.geometry.y, mall_gdf.geometry.x))
mall_names = mall_gdf['mall_name'].tolist()
mall_postals = mall_gdf['postal'].tolist()

tree = cKDTree(mall_coords)

results = []

print(f"--- starting sample test for {len(postal_gdf_sample)} postals ---")

# --- 2. sample loop ---
for idx, row in postal_gdf_sample.iterrows():
    p_code = str(row['postal_code'])
    origin_coords = (row.geometry.y, row.geometry.x)
    
    print(f"\nchecking postal: {p_code} at {origin_coords}")
    
    # get 3 closest malls by straight line
    dist, indices = tree.query(origin_coords, k=3) 
    
    best_walk = float('inf')
    closest_mall = None

    for i in indices:
        dest_coords = mall_coords[i]
        mall_name = mall_names[i]
        mall_postal = mall_postals[i]
        
        # using the routingsvc endpoint from the docs
        url = f"https://www.onemap.gov.sg/api/public/routingsvc/route?start={origin_coords[0]},{origin_coords[1]}&end={dest_coords[0]},{dest_coords[1]}&routeType=walk"
        
        try:

            token = onemap_api_key if onemap_api_key.startswith("Bearer ") else f"Bearer {onemap_api_key}"
            headers = {"Authorization": token}
            # headers = {"Authorization": onemap_api_key}
            res = requests.get(url, headers=headers, timeout=10)
            
            if res.status_code == 200:
                data = res.json()
                if 'route_summary' in data:
                    walk_m = data['route_summary']['total_distance']
                    print(f"  -> found route to {mall_name}: {walk_m}m")
                    if walk_m < best_walk:
                        best_walk = walk_m
                        closest_mall = mall_name
                        closest_mall_postal= mall_postal
                else:
                    print(f"  -> no route summary found for {mall_name}")
            else:
                print(f"  -> api error {res.status_code}: {res.text}")
            
            time.sleep(0.2) # slightly longer sleep for the sample
        except Exception as e:
            print(f"  -> request failed: {e}")

    results.append({
        "postal_code": p_code,
        "nearest_mall": closest_mall,
        "nearest_mall_postal": closest_mall_postal,
        "walking_dist_m": best_walk if best_walk != float('inf') else None
    })

# --- 3. display output ---
sample_df = pd.DataFrame(results)
print("\n--- final sample output ---")
print(sample_df)

# check if we actually got results
if sample_df['walking_dist_m'].isnull().all():
    print("\nwarning: all walking distances are null. check your api key and coordinate order.")
else:
    print("\nsuccess: distances found! you can now run the full script.")

--- starting sample test for 5 postals ---

checking postal: 560225 at (1.3673961277036402, 103.83815000747873)
  -> found route to AMK HUB: 1466m
  -> found route to THOMSON PLAZA: 3220m
  -> found route to JUNCTION 8: 2982m

checking postal: 560174 at (1.3751965588189876, 103.83769584780252)
  -> found route to AMK HUB: 1902m
  -> found route to THOMSON PLAZA: 3640m
  -> found route to JUNCTION 8: 3980m

checking postal: 560440 at (1.3664278983686127, 103.85431149122876)
  -> found route to AMK HUB: 1020m
  -> found route to JUNCTION 8: 2376m
  -> found route to NEX: 3290m

checking postal: 560637 at (1.3803292495669066, 103.8423366937204)
  -> found route to AMK HUB: 1816m
  -> found route to THOMSON PLAZA: 4542m
  -> found route to JUNCTION 8: 4169m

checking postal: 560571 at (1.3701655201821288, 103.85494058284989)
  -> found route to AMK HUB: 860m
  -> found route to JUNCTION 8: 2846m
  -> found route to NEX: 3812m

--- final sample output ---
  postal_code nearest_mall nearest_

3.2 Run full script if results generated for the sample postal codes

In [32]:
# --- 1. Load Datasets (full) ---
raw_hdb_gdf = gpd.read_file(hdb_points_file).to_crs(epsg=4326)
postal_gdf = raw_hdb_gdf.drop_duplicates(subset=['postal_code']).copy()

mall_gdf = mall_gdf.to_crs(epsg=4326)
mall_coords = list(zip(mall_gdf.geometry.y, mall_gdf.geometry.x))
mall_names = mall_gdf['mall_name'].tolist()
mall_postals = mall_gdf['postal'].tolist()
tree = cKDTree(mall_coords)

# --- 2. Resume logic ---
if os.path.exists(output_file):
    processed_df = pd.read_csv(output_file)
    processed_postals = set(processed_df['postal_code'].astype(str))
    results = processed_df.to_dict('records')
    print(f"Resuming... {len(processed_postals)} postals already completed.")
else:
    results = []
    processed_postals = set()

# --- 3. Full loop ---
print(f"Starting processing for {len(postal_gdf) - len(processed_postals)} remaining postals...")

try:
    for idx, row in postal_gdf.iterrows():
        p_code = row['postal_code']
        if p_code in processed_postals:
            continue

        origin_coords = (row.geometry.y, row.geometry.x)
        dist, indices = tree.query(origin_coords, k=3)
        
        closest_walk = float('inf')
        closest_mall = None
        closest_mall_postal = None

        for i in indices:
            dest_coords = mall_coords[i]
            token = onemap_api_key if onemap_api_key.startswith("Bearer ") else f"Bearer {onemap_api_key}"
            url = f"https://www.onemap.gov.sg/api/public/routingsvc/route?start={origin_coords[0]},{origin_coords[1]}&end={dest_coords[0]},{dest_coords[1]}&routeType=walk"
            
            try:
                res = requests.get(url, headers={"Authorization": token}, timeout=10)
                if res.status_code == 200:
                    data = res.json()
                    if 'route_summary' in data:
                        walk_m = data['route_summary']['total_distance']
                        if walk_m < closest_walk:
                            closest_walk = walk_m
                            closest_mall = mall_names[i]
                            closest_mall_postal = mall_postals[i]
                time.sleep(0.2)
            except:
                continue

        results.append({
            "postal_code": p_code,
            "nearest_mall": closest_mall,
            "nearest_mall_postal": closest_mall_postal,
            "walking_dist_m": closest_walk if closest_walk != float('inf') else None
        })
        processed_postals.add(p_code)

        if len(results) % 50 == 0:
            pd.DataFrame(results).to_csv(output_file, index=False)
            print(f"progress: {len(results)} / {len(postal_gdf)} unique postals saved.")
            

except KeyboardInterrupt:
    print("\nPaused. Saving current progress...")

# Final save
pd.DataFrame(results).to_csv(output_file, index=False)
print(f"Task complete! Results saved to {output_file}")

Starting processing for 9650 remaining postals...
progress: 50 / 9650 unique postals saved.
progress: 100 / 9650 unique postals saved.
progress: 150 / 9650 unique postals saved.
progress: 200 / 9650 unique postals saved.
progress: 250 / 9650 unique postals saved.
progress: 300 / 9650 unique postals saved.
progress: 350 / 9650 unique postals saved.
progress: 400 / 9650 unique postals saved.
progress: 450 / 9650 unique postals saved.
progress: 500 / 9650 unique postals saved.
progress: 550 / 9650 unique postals saved.
progress: 600 / 9650 unique postals saved.
progress: 650 / 9650 unique postals saved.
progress: 700 / 9650 unique postals saved.
progress: 750 / 9650 unique postals saved.
progress: 800 / 9650 unique postals saved.
progress: 850 / 9650 unique postals saved.
progress: 900 / 9650 unique postals saved.
progress: 950 / 9650 unique postals saved.
progress: 1000 / 9650 unique postals saved.
progress: 1050 / 9650 unique postals saved.
progress: 1100 / 9650 unique postals saved.
pr

### 4. Data integrity checks, save results to file

In [88]:
full_hdb_point = gpd.read_file("resale_2019_to_2025_cleaned.geojson")
hdb_mall_distances = pd.read_csv("hdb_mall_distances.csv")

In [ ]:
# 1. Force the postal_code in the GeoDataFrame to be a string
full_hdb_point['postal_code'] = full_hdb_point['postal_code'].astype(str).str.zfill(6)

# 2. Force the postal_code in the distance CSV to be a string
hdb_mall_distances['postal_code'] = hdb_mall_distances['postal_code'].astype(str).str.zfill(6)

Merge complete!


4.1 Checking for any anomalies, then verify on onemap if walking distance is too big

In [90]:
print(hdb_mall_distances)
hdb_mall_distances.info()
hdb_mall_distances.isnull().any()

      postal_code                        nearest_mall  nearest_mall_postal  \
0          560225                             AMK HUB               569933   
1          560174                             AMK HUB               569933   
2          560440                             AMK HUB               569933   
3          560637                             AMK HUB               569933   
4          560571                             AMK HUB               569933   
...           ...                                 ...                  ...   
9645       763478  CHURCH OF OUR LADY STAR OF THE SEA               768579   
9646       761478  CHURCH OF OUR LADY STAR OF THE SEA               768579   
9647       761795                     NORTHPOINT CITY               769098   
9648       762479  CHURCH OF OUR LADY STAR OF THE SEA               768579   
9649       763479  CHURCH OF OUR LADY STAR OF THE SEA               768579   

      walking_dist_m  
0               1466  
1               1

postal_code            False
nearest_mall           False
nearest_mall_postal    False
walking_dist_m         False
dtype: bool

In [ ]:
hdb_mall_distances[hdb_mall_distances["walking_dist_m"] > 2500]

,month,town,flat_type,block,street_name,address,postal_code,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price,geometry,nearest_hawker,nearest_hawker_postal,walking_dist_m
2657,2025-01-01,BUKIT BATOK,3 ROOM,263,BT BATOK EAST AVE 4,263 BUKIT BATOK EAST AVENUE 4,650263,07 TO 09,73.0,Model A,1985,59 years 06 months,417000.0,POINT (103.75909 1.35029),Bukit Batok West Hawker Centre,651469.0,2544.0
2658,2025-01-01,BUKIT BATOK,3 ROOM,264,BT BATOK EAST AVE 4,264 BUKIT BATOK EAST AVENUE 4,650264,10 TO 12,73.0,Model A,1985,59 years 05 months,320000.0,POINT (103.75904 1.34991),Bukit Timah Interim Hawker Centre and Market,599213.0,2510.0
2659,2025-01-01,BUKIT BATOK,3 ROOM,263,BT BATOK EAST AVE 4,263 BUKIT BATOK EAST AVENUE 4,650263,04 TO 06,73.0,Model A,1985,59 years 06 months,410000.0,POINT (103.75909 1.35029),Bukit Batok West Hawker Centre,651469.0,2544.0
2663,2025-04-01,BUKIT BATOK,3 ROOM,264,BT BATOK EAST AVE 4,264 BUKIT BATOK EAST AVENUE 4,650264,04 TO 06,73.0,Model A,1985,59 years 01 month,418000.0,POINT (103.75904 1.34991),Bukit Timah Interim Hawker Centre and Market,599213.0,2510.0
2664,2025-04-01,BUKIT BATOK,3 ROOM,263,BT BATOK EAST AVE 4,263 BUKIT BATOK EAST AVENUE 4,650263,07 TO 09,73.0,Model A,1985,59 years 03 months,420000.0,POINT (103.75909 1.35029),Bukit Batok West Hawker Centre,651469.0,2544.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21524,2025-03-01,TAMPINES,EXECUTIVE,498B,TAMPINES ST 45,498B TAMPINES STREET 45,521498,07 TO 09,147.0,Premium Maisonette,1997,71 years 05 months,1050000.0,POINT (103.9586 1.3611),Pasir Ris Central Hawker Centre,519641.0,2605.0
21525,2025-04-01,TAMPINES,EXECUTIVE,497D,TAMPINES ST 45,497D TAMPINES STREET 45,523497,04 TO 06,139.0,Premium Apartment,1996,70 years 03 months,928000.0,POINT (103.95908 1.35883),Hawker Centre @ Our Tampines Hub,528523.0,2618.0
21527,2025-06-01,TAMPINES,EXECUTIVE,497J,TAMPINES ST 45,497J TAMPINES STREET 45,527497,10 TO 12,144.0,Premium Apartment,1996,70 years,968000.0,POINT (103.9595 1.35926),Hawker Centre @ Our Tampines Hub,528523.0,2695.0
21528,2025-07-01,TAMPINES,EXECUTIVE,498F,TAMPINES ST 45,498F TAMPINES STREET 45,524498,07 TO 09,147.0,Premium Maisonette,1997,71 years,1000000.0,POINT (103.95856 1.36191),Pasir Ris Central Hawker Centre,519641.0,2536.0


In [ ]:
len(hdb_mall_distances)
hdb_mall_distances = pd.read_csv("hdb_mall_distances.csv")

9650